In [ ]:
import myModule as mine
import time
import shutil
import numpy as np
import matplotlib.pyplot as plt
# import Unified_cl_methods as ucm
import MODE_utilities as AS
import Case1 as c1
import pygad
import os
from scipy.stats import zscore

In [ ]:
# tipos = "RAM"
# type_of_data = os.path.join("Experiment_3", tipos)
# # My_data = mine.read_csv("RAM")
# My_data = mine.read_csv(type_of_data, tipos)
# My_data = My_data[:41000]
# print(My_data.shape)

# # Assuming My_data is already defined and is a flat NumPy array
# steps = 500

# # Keep only the largest multiple of 2000
# num_full_series = len(My_data) // steps
# trimmed_data = My_data[:num_full_series * steps]

# # Reshape to (num_series, steps)
# split_data = trimmed_data.reshape((num_full_series, steps))
# split_data = np.delete(split_data, 7, axis=0)
# print(split_data.shape)  # Output: (num_series, 2000)

# split_percentage = 0.8
# offline_data = split_data[:, :int(split_percentage*split_data.shape[1])]
# print(offline_data.shape)
# ''' All cases will be tested on this data set, for a fair comparison'''
# online_data = split_data[:, int(split_percentage*split_data.shape[1]):]
# print(online_data.shape)


# Raspberry Pi Devices

In [ ]:
TYPE_OF_DATA = 'memory'
SPLIT = 0.80
split_data = AS.read_raspberry_pi_dataset(
    series_length=500, column=TYPE_OF_DATA
)

split_data = split_data[:60][:]

split_data


offline_data = split_data[:, :int(SPLIT*split_data.shape[1])]
online_data = split_data[:, int(SPLIT*split_data.shape[1]):]

In [ ]:
# Step 1: scale
X_scaled = c1.minmax_scale(offline_data)
D = c1.compute_DTW_matrix(X_scaled)
opt_k = c1.elbow_method_kmeans(D)

# clustering = c1.K_DTW(K_Clusters=3, max_iter=100, tol=1e-6)
# mem, centroids = clustering.fit(X_scaled)


In [ ]:
clustering = c1.K_DTW(K_Clusters=opt_k, max_iter=50, tol=1e-4)
mem, centroids = clustering.fit(X_scaled)


In [ ]:
# mem = array of cluster labels for each time series
cluster_dict = {int(u): [] for u in np.unique(mem)}

for idx, label in enumerate(mem):
    cluster_dict[label].append(idx)


In [ ]:
cluster_dict

In [ ]:

cluster_concat_dict = {label: np.concatenate([X_scaled[i] for i in indices]) 
                       for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_concat_dict.items():
    print(f"Cluster {label}: concatenated length = {data.shape[0]}")
    # print(data.shape)
    
    # plt.show()

In [ ]:
cluster_ONLINE_data_dict = {label: online_data[indices] for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_ONLINE_data_dict.items():
    print(f"Online Cluster {label}: shape {data.shape}")
    # print(type(data[0]))

In [ ]:
gene_space = [
    {'low': 2, 'high': 10, 'step': 1},   # hidden_dim
    {'low': 1, 'high': 2, 'step': 1},     # layer_dim
    {'low': 5, 'high': 20, 'step': 1},    # window
    {'low': 8, 'high': 32, 'step': 1},      # batch_size
    {'low': 10, 'high': 50, 'step': 1},      # epochs
    {'low': 0.0001, 'high': 0.5},           # learning_rate
]


In [ ]:

class MPC_GA:
    def __init__(self, gene_space,
                #   online_ts,
                  cluster_concat_dict,
                  cluster_ONLINE_data_dict,
                  cluster_dict):
        # self.offline_ts = offline_ts
        self.DATA = cluster_concat_dict
        self.TTST = cluster_ONLINE_data_dict
        self.gene_space = gene_space
        # self.online_ts = online_ts
        self.ids = cluster_dict
        # self.type_of_data = type_of_data
        
        for label, data in self.DATA.items():
            start = time.time()
            self.dedomena = data
            print(f"Optimizing Cluster no. {label}:\nConcatenated length = {data.shape[0]}\nIDs: {self.ids[label]}")
            self.best_model = None
            self.best_solution = None
            self.best_fitness = float("inf")
            self.best_instance = None
            self.result = []
            self.training_duration = None
            self.run()
            duration = time.time() - start
            print(f"Cluster no. {label} took {duration} [sec]")
            self.evaluate_online(label)




    def fitness_func(self, ga_instance, genes, genes_idx):
        chromosome = {
            'hidden_dim': int(genes[0]),
            'layer_dim': int(genes[1]),
            'window': int(genes[2]),
            'batch_size': int(genes[3]),
            'epochs': int(genes[4]),
            'learning_rate': genes[5]
        }
        
        instance = c1.Instance(self.dedomena, chromosome)
        instance.train_phase()
        rmse, _, _, _ = instance.validation_phase()

        if rmse < self.best_fitness:
            self.best_fitness = rmse
            self.best_solution = genes.copy()
            self.best_model = instance.model
            self.best_instance = instance
        return -rmse
    

    def run(self, generations=11, population_size=10):
        ga_instance = pygad.GA(
            num_generations=generations,
            num_parents_mating=10,
            fitness_func=self.fitness_func,
            sol_per_pop=population_size,
            num_genes=len(self.gene_space),
            gene_space=self.gene_space,
            mutation_percent_genes=50,
            mutation_type="random",
            crossover_type="single_point",
            allow_duplicate_genes=True,
            keep_elitism=2
        )
       
        ga_instance.run()
       

    def evaluate_online(self, label):
        
        save_dir = f"MPC_Rasbery_Pi_memory"

        if os.path.exists(save_dir) == False:
            os.makedirs(save_dir)
            
        for i in range(len(self.TTST[label])):

            rmse, r2, _, preds, targs = c1.evaluate_on_new_data(
                trained_model=self.best_instance.model,
                new_data=self.TTST[label][i],
                chromosome=self.best_instance.chromosome
            )
        # print(f"Online RMSE: {rmse:.4f}| latency: {latency:.4f}| stats: {stats}")
            
            with open(f'MPC_Rasbery_Pi_memory.txt', 'a') as f:
                f.write(f"Devices: {self.ids[label][i]} | RMSE: {rmse.item()}\n")

        # Save prediction plot
            plt.figure(figsize=(12, 8))
            plt.plot(preds, label='Predictions', color='royalblue', linestyle='--', marker='o')
            plt.plot(targs, label='Targets', color='crimson', linestyle='-', marker='x')
            plt.xlabel('Time'); plt.ylabel('Value'); plt.grid(True)
            plt.title(f'Testing Phase - device {self.ids[label][i]}')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"{self.ids[label][i]}.png"))
            plt.close()
        
    


In [ ]:
with open(f'MPC_Rasbery_Pi_memory.txt', 'w') as f:
    f.write(f"General info\n=============\n")


In [ ]:
MPC_GA(gene_space,
        cluster_concat_dict,
        cluster_ONLINE_data_dict,
        cluster_dict)

# Experiment 3 CPU

In [ ]:
tipos = "CPU"
type_of_data = os.path.join("Experiment_3", tipos)
# My_data = mine.read_csv("CPU")
My_data = mine.read_csv(type_of_data, tipos)

print(My_data.shape)

# Assuming My_data is already defined and is a flat NumPy array
steps = 500

# Keep only the largest multiple of 2000
num_full_series = len(My_data) // steps
trimmed_data = My_data[:num_full_series * steps]

# Reshape to (num_series, steps)
split_data = trimmed_data.reshape((num_full_series, steps))
split_data = np.delete(split_data, 7, axis=0)
print(split_data.shape)  # Output: (num_series, 2000)

split_percentage = 0.8
offline_data = split_data[:, :int(split_percentage*split_data.shape[1])]
print(offline_data.shape)
''' All cases will be tested on this data set, for a fair comparison'''
online_data = split_data[:, int(split_percentage*split_data.shape[1]):]
print(online_data.shape)


# Step 1: scale
X_scaled = c1.minmax_scale(offline_data)
D = c1.compute_DTW_matrix(X_scaled)
opt_k = c1.elbow_method_kmeans(D)

# clustering = c1.K_DTW(K_Clusters=3, max_iter=100, tol=1e-6)
# mem, centroids = clustering.fit(X_scaled)


clustering = c1.K_DTW(K_Clusters=opt_k, max_iter=50, tol=1e-4)
mem, centroids = clustering.fit(X_scaled)


# mem = array of cluster labels for each time series
cluster_dict = {int(u): [] for u in np.unique(mem)}

for idx, label in enumerate(mem):
    cluster_dict[label].append(idx)



cluster_concat_dict = {label: np.concatenate([X_scaled[i] for i in indices]) 
                       for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_concat_dict.items():
    print(f"Cluster {label}: concatenated length = {data.shape[0]}")
    # print(data.shape)
    
    # plt.show()

cluster_ONLINE_data_dict = {label: online_data[indices] for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_ONLINE_data_dict.items():
    print(f"Online Cluster {label}: shape {data.shape}")
    # print(type(data[0]))

gene_space = [
    {'low': 2, 'high': 10, 'step': 1},   # hidden_dim
    {'low': 1, 'high': 2, 'step': 1},     # layer_dim
    {'low': 5, 'high': 20, 'step': 1},    # window
    {'low': 8, 'high': 32, 'step': 1},      # batch_size
    {'low': 10, 'high': 50, 'step': 1},      # epochs
    {'low': 0.0001, 'high': 0.5},           # learning_rate
]



class MPC_GA:
    def __init__(self, gene_space,
                #   online_ts,
                  cluster_concat_dict,
                  cluster_ONLINE_data_dict,
                  cluster_dict,
                  type_of_data):
        # self.offline_ts = offline_ts
        self.DATA = cluster_concat_dict
        self.TTST = cluster_ONLINE_data_dict
        self.gene_space = gene_space
        # self.online_ts = online_ts
        self.ids = cluster_dict
        self.type_of_data = type_of_data
        
        for label, data in self.DATA.items():
            start = time.time()
            self.dedomena = data
            print(f"Optimizing Cluster no. {label}:\nConcatenated length = {data.shape[0]}\nIDs: {self.ids[label]}")
            self.best_model = None
            self.best_solution = None
            self.best_fitness = float("inf")
            self.best_instance = None
            self.result = []
            self.training_duration = None
            self.run()
            duration = time.time() - start
            print(f"Cluster no. {label} took {duration} [sec]")
            self.evaluate_online(label)




    def fitness_func(self, ga_instance, genes, genes_idx):
        chromosome = {
            'hidden_dim': int(genes[0]),
            'layer_dim': int(genes[1]),
            'window': int(genes[2]),
            'batch_size': int(genes[3]),
            'epochs': int(genes[4]),
            'learning_rate': genes[5]
        }
        
        instance = c1.Instance(self.dedomena, chromosome)
        instance.train_phase()
        rmse, _, _, _ = instance.validation_phase()

        if rmse < self.best_fitness:
            self.best_fitness = rmse
            self.best_solution = genes.copy()
            self.best_model = instance.model
            self.best_instance = instance
        return -rmse
    

    def run(self, generations=10, population_size=20):
        ga_instance = pygad.GA(
            num_generations=generations,
            num_parents_mating=10,
            fitness_func=self.fitness_func,
            sol_per_pop=population_size,
            num_genes=len(self.gene_space),
            gene_space=self.gene_space,
            mutation_percent_genes=50,
            mutation_type="random",
            crossover_type="single_point",
            allow_duplicate_genes=True,
            keep_elitism=10
        )
       
        ga_instance.run()
       

    def evaluate_online(self, label):
        
        save_dir = f"MPC_exp_3_cpu"

        if os.path.exists(save_dir) == False:
            os.makedirs(save_dir)
            
        for i in range(len(self.TTST[label])):

            rmse, r2, _, preds, targs = c1.evaluate_on_new_data(
                trained_model=self.best_instance.model,
                new_data=self.TTST[label][i],
                chromosome=self.best_instance.chromosome
            )
        # print(f"Online RMSE: {rmse:.4f}| latency: {latency:.4f}| stats: {stats}")
            
            with open(f'MPC_res_exp3_cpu.txt', 'a') as f:
                f.write(f"Devices: {self.ids[label][i]} | RMSE: {rmse.item()} | R^2: {r2.item()}\n")

        # Save prediction plot
            plt.figure(figsize=(12, 8))
            plt.plot(preds, label='Predictions', color='royalblue', linestyle='--', marker='o')
            plt.plot(targs, label='Targets', color='crimson', linestyle='-', marker='x')
            plt.xlabel('Time'); plt.ylabel('Value'); plt.grid(True)
            plt.title(f'Testing Phase - device {self.ids[label][i]}')
            plt.legend()
            plt.tight_layout()
            plt.savefig(os.path.join(save_dir, f"{self.ids[label][i]}.png"))
            plt.close()
        
    

with open(f'MPC_res_exp3_cpu.txt', 'w') as f:
    f.write(f"General info\n=============\n")



MPC_GA(gene_space,
        cluster_concat_dict,
        cluster_ONLINE_data_dict,
        cluster_dict,
        type_of_data)









# CPU

In [ ]:
tipos = "CPU"
type_of_data = os.path.join("Experiment_3", tipos)
# My_data = mine.read_csv("RAM")
My_data = mine.read_csv(type_of_data, tipos)
print(My_data.shape)

# Assuming My_data is already defined and is a flat NumPy array
steps = 500

# Keep only the largest multiple of 2000
num_full_series = len(My_data) // steps
trimmed_data = My_data[:num_full_series * steps]

# Reshape to (num_series, steps)
split_data = trimmed_data.reshape((num_full_series, steps))
# split_data = np.delete(split_data, 7, axis=0)
print(split_data.shape)  # Output: (num_series, 2000)

split_percentage = 0.8
offline_data = split_data[:, :int(split_percentage*split_data.shape[1])]
print(offline_data.shape)
''' All cases will be tested on this data set, for a fair comparison'''
online_data = split_data[:, int(split_percentage*split_data.shape[1]):]
print(online_data.shape)





# Step 1: scale
X_scaled = c1.minmax_scale(offline_data)
D = c1.compute_DTW_matrix(X_scaled)
opt_k = c1.elbow_method_kmeans(D)

# clustering = c1.K_DTW(K_Clusters=3, max_iter=100, tol=1e-6)
# mem, centroids = clustering.fit(X_scaled)


clustering = c1.K_DTW(K_Clusters=opt_k, max_iter=50, tol=1e-4)
mem, centroids = clustering.fit(X_scaled)

# mem = array of cluster labels for each time series
cluster_dict = {int(u): [] for u in np.unique(mem)}

for idx, label in enumerate(mem):
    cluster_dict[label].append(idx)

cluster_concat_dict = {label: np.concatenate([X_scaled[i] for i in indices]) 
                       for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_concat_dict.items():
    print(f"Cluster {label}: concatenated length = {data.shape[0]}")
    # print(data.shape)
    
    # plt.show()

cluster_ONLINE_data_dict = {label: online_data[indices] for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_ONLINE_data_dict.items():
    print(f"Online Cluster {label}: shape {data.shape}")
    # print(type(data[0]))


with open(f'MPC_res_exp3_cpu.txt', 'w') as f:
    f.write(f"General info\n=============\n")

MPC_GA(gene_space,
        cluster_concat_dict,
        cluster_ONLINE_data_dict,
        cluster_dict,
        type_of_data)







# touse RAM

In [ ]:
type_of_data = "touse_RAM"
tipos = "RAM"
# My_data = mine.read_csv("RAM")
My_data = mine.read_csv(type_of_data, tipos)
print(My_data.shape)

# Assuming My_data is already defined and is a flat NumPy array
steps = 500

# Keep only the largest multiple of 2000
num_full_series = len(My_data) // steps
trimmed_data = My_data[:num_full_series * steps]

# Reshape to (num_series, steps)
split_data = trimmed_data.reshape((num_full_series, steps))
# split_data = np.delete(split_data, 7, axis=0)
print(split_data.shape)  # Output: (num_series, 2000)

split_percentage = 0.8
offline_data = split_data[:, :int(split_percentage*split_data.shape[1])]
print(offline_data.shape)
''' All cases will be tested on this data set, for a fair comparison'''
online_data = split_data[:, int(split_percentage*split_data.shape[1]):]
print(online_data.shape)





# Step 1: scale
X_scaled = c1.minmax_scale(offline_data)
D = c1.compute_DTW_matrix(X_scaled)
opt_k = c1.elbow_method_kmeans(D)

# clustering = c1.K_DTW(K_Clusters=3, max_iter=100, tol=1e-6)
# mem, centroids = clustering.fit(X_scaled)


clustering = c1.K_DTW(K_Clusters=opt_k, max_iter=50, tol=1e-4)
mem, centroids = clustering.fit(X_scaled)

# mem = array of cluster labels for each time series
cluster_dict = {int(u): [] for u in np.unique(mem)}

for idx, label in enumerate(mem):
    cluster_dict[label].append(idx)

cluster_concat_dict = {label: np.concatenate([X_scaled[i] for i in indices]) 
                       for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_concat_dict.items():
    print(f"Cluster {label}: concatenated length = {data.shape[0]}")
    # print(data.shape)
    
    # plt.show()

cluster_ONLINE_data_dict = {label: online_data[indices] for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_ONLINE_data_dict.items():
    print(f"Online Cluster {label}: shape {data.shape}")
    # print(type(data[0]))


with open(f'MPC_res_{type_of_data}.txt', 'w') as f:
    f.write(f"General info\n=============\n")

MPC_GA(gene_space,
        cluster_concat_dict,
        cluster_ONLINE_data_dict,
        cluster_dict,
        type_of_data)







# touse CPU

In [ ]:
type_of_data = "touse_CPU"
tipos = "CPU"
# My_data = mine.read_csv("RAM")
My_data = mine.read_csv(type_of_data, tipos)
print(My_data.shape)

# Assuming My_data is already defined and is a flat NumPy array
steps = 500

# Keep only the largest multiple of 2000
num_full_series = len(My_data) // steps
trimmed_data = My_data[:num_full_series * steps]

# Reshape to (num_series, steps)
split_data = trimmed_data.reshape((num_full_series, steps))
split_data = np.delete(split_data, 7, axis=0)
print(split_data.shape)  # Output: (num_series, 2000)

split_percentage = 0.8
offline_data = split_data[:, :int(split_percentage*split_data.shape[1])]
print(offline_data.shape)
''' All cases will be tested on this data set, for a fair comparison'''
online_data = split_data[:, int(split_percentage*split_data.shape[1]):]
print(online_data.shape)





# Step 1: scale
X_scaled = c1.minmax_scale(offline_data)
D = c1.compute_DTW_matrix(X_scaled)
opt_k = c1.elbow_method_kmeans(D)

# clustering = c1.K_DTW(K_Clusters=3, max_iter=100, tol=1e-6)
# mem, centroids = clustering.fit(X_scaled)


clustering = c1.K_DTW(K_Clusters=opt_k, max_iter=50, tol=1e-4)
mem, centroids = clustering.fit(X_scaled)

# mem = array of cluster labels for each time series
cluster_dict = {int(u): [] for u in np.unique(mem)}

for idx, label in enumerate(mem):
    cluster_dict[label].append(idx)

cluster_concat_dict = {label: np.concatenate([X_scaled[i] for i in indices]) 
                       for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_concat_dict.items():
    print(f"Cluster {label}: concatenated length = {data.shape[0]}")
    # print(data.shape)
    
    # plt.show()

cluster_ONLINE_data_dict = {label: online_data[indices] for label, indices in cluster_dict.items()}

# Check one example
for label, data in cluster_ONLINE_data_dict.items():
    print(f"Online Cluster {label}: shape {data.shape}")
    # print(type(data[0]))


with open(f'MPC_res_{type_of_data}.txt', 'w') as f:
    f.write(f"General info\n=============\n")

MPC_GA(gene_space,
        cluster_concat_dict,
        cluster_ONLINE_data_dict,
        cluster_dict,
        type_of_data)





